In [ ]:
from google.colab import files

files.upload()

In [ ]:
from google.colab import files

files.upload()

In [ ]:
from google.colab import files

files.upload()

In [ ]:
import joblib
import pandas as pd

model = joblib.load("demand_model.pkl")

features = joblib.load(
    "model_features.pkl"
)

df = pd.read_csv(
    "dynamic_pricing_processed.csv"
)

product_state = df.iloc[0].to_dict()

In [ ]:
import random
from datetime import datetime, UTC


EVENT_TYPES = [
    "PRODUCT_VIEW",
    "ADD_TO_CART",
    "PURCHASE",
    "COMPETITOR_PRICE_CHANGE",
    "INVENTORY_UPDATE"
]


def generate_event(product_id):

    event_type = random.choice(EVENT_TYPES)

    event = {
        "event_id": f"EVT-{random.randint(100000, 999999)}",
        "event_type": event_type,
        "product_id": product_id,
        "timestamp": datetime.now(UTC).isoformat()
    }

    if event_type == "COMPETITOR_PRICE_CHANGE":
        event["price_change_percentage"] = round(
            random.uniform(-0.10, 0.10),
            3
        )

    elif event_type == "INVENTORY_UPDATE":
        event["inventory_change"] = random.randint(
            -20,
            20
        )

    return event

In [ ]:
def update_product_state(product_state, event):

    state = product_state.copy()

    event_type = event["event_type"]

    if event_type == "PRODUCT_VIEW":
        state["views_last_hour"] += 1

    elif event_type == "ADD_TO_CART":
        state["add_to_cart_count"] += 1

    elif event_type == "PURCHASE":
        state["inventory_level"] = max(
            0,
            state["inventory_level"] - 1
        )

    elif event_type == "COMPETITOR_PRICE_CHANGE":

        change = event["price_change_percentage"]

        state["competitor_price"] = (
            state["competitor_price"] * (1 + change)
        )

    elif event_type == "INVENTORY_UPDATE":

        state["inventory_level"] = max(
            0,
            state["inventory_level"]
            + event["inventory_change"]
        )

    return state

In [ ]:
def generate_pricing_decision(
    product_data,
    model,
    features
):

    best_result, results_df = optimize_price(
        product_data,
        model,
        features
    )

    explanation = explain_pricing_decision(
        current_price=product_data["current_price"],
        recommended_price=best_result["recommended_price"],
        competitor_price=product_data["competitor_price"],
        inventory_level=product_data["inventory_level"],
        predicted_demand=best_result["predicted_demand"]
    )

    decision = {
        "product_id": product_data["product_id"],
        "current_price": product_data["current_price"],
        "recommended_price": best_result["recommended_price"],
        "predicted_demand": best_result["predicted_demand"],
        "expected_revenue": best_result["expected_revenue"],
        "expected_profit": best_result["expected_profit"],
        "action": explanation["action"],
        "price_change_percentage": explanation[
            "price_change_percentage"
        ],
        "reasons": explanation["reasons"]
    }

    return decision, results_df

In [ ]:
import numpy as np
import pandas as pd


def calculate_profit(price, cost_price, demand):

    return (price - cost_price) * demand


def apply_guardrails(
    recommended_price,
    current_price,
    cost_price,
    competitor_price,
    min_margin=0.10,
    max_price_change=0.20,
    max_market_multiplier=1.25
):

    # Minimum price based on cost and minimum margin
    min_profit_price = cost_price * (1 + min_margin)

    # Maximum allowed decrease
    min_change_price = current_price * (
        1 - max_price_change
    )

    # Maximum allowed increase
    max_change_price = current_price * (
        1 + max_price_change
    )

    # Maximum market-based price
    max_market_price = (
        competitor_price * max_market_multiplier
    )

    # Apply minimum constraints
    final_price = max(
        recommended_price,
        min_profit_price,
        min_change_price
    )

    # Apply maximum constraints
    final_price = min(
        final_price,
        max_change_price,
        max_market_price
    )

    return round(float(final_price), 2)


def optimize_price(
    product_data,
    model,
    features,
    min_margin=0.10,
    max_price_increase=0.20,
    max_price_decrease=0.20,
    num_candidates=21
):

    current_price = float(
        product_data["current_price"]
    )

    cost_price = float(
        product_data["cost_price"]
    )

    competitor_price = float(
        product_data["competitor_price"]
    )

    # Generate allowed price range
    min_price = max(
        cost_price * (1 + min_margin),
        current_price * (
            1 - max_price_decrease
        )
    )

    max_price = current_price * (
        1 + max_price_increase
    )

    candidate_prices = np.linspace(
        min_price,
        max_price,
        num_candidates
    )

    results = []

    for candidate_price in candidate_prices:

        temp = product_data.copy()

        # Update candidate price
        temp["current_price"] = float(
            candidate_price
        )

        # Recalculate price-dependent features
        temp["price_ratio_to_competitor"] = (
            candidate_price /
            max(temp["competitor_price"], 0.01)
        )

        temp["price_difference"] = (
            candidate_price -
            temp["competitor_price"]
        )

        temp["profit_margin"] = (
            (candidate_price - temp["cost_price"]) /
            max(candidate_price, 0.01)
        )

        # Convert to DataFrame
        input_df = pd.DataFrame([temp])

        # Select exact model features
        input_df = input_df[features]

        # Predict demand
        predicted_demand = max(
            0,
            float(model.predict(input_df)[0])
        )

        # Calculate business metrics
        expected_revenue = (
            candidate_price *
            predicted_demand
        )

        expected_profit = calculate_profit(
            candidate_price,
            cost_price,
            predicted_demand
        )

        results.append({

            "candidate_price": round(
                float(candidate_price),
                2
            ),

            "predicted_demand": round(
                predicted_demand,
                2
            ),

            "expected_revenue": round(
                expected_revenue,
                2
            ),

            "expected_profit": round(
                expected_profit,
                2
            )
        })

    results_df = pd.DataFrame(results)

    # Find maximum expected profit
    best_result = results_df.loc[
        results_df[
            "expected_profit"
        ].idxmax()
    ].to_dict()

    # Apply business guardrails
    guarded_price = apply_guardrails(

        recommended_price=
        best_result["candidate_price"],

        current_price=current_price,

        cost_price=cost_price,

        competitor_price=competitor_price,

        min_margin=min_margin,

        max_price_change=
        max_price_increase
    )

    best_result[
        "recommended_price"
    ] = guarded_price

    return best_result, results_df

In [ ]:
def explain_pricing_decision(
    current_price,
    recommended_price,
    competitor_price,
    inventory_level,
    predicted_demand
):

    reasons = []

    price_change = (
        recommended_price - current_price
    ) / current_price

    if predicted_demand > 20:

        reasons.append(
            "High predicted demand"
        )

    elif predicted_demand < 5:

        reasons.append(
            "Low predicted demand"
        )

    if inventory_level < 30:

        reasons.append(
            "Low inventory level"
        )

    elif inventory_level > 200:

        reasons.append(
            "High inventory level"
        )

    if competitor_price > current_price:

        reasons.append(
            "Competitor price is higher"
        )

    elif competitor_price < current_price:

        reasons.append(
            "Competitor price is lower"
        )

    if price_change > 0.01:

        action = "INCREASE_PRICE"

    elif price_change < -0.01:

        action = "DECREASE_PRICE"

    else:

        action = "KEEP_PRICE"

    return {

        "action": action,

        "price_change_percentage": round(
            price_change * 100,
            2
        ),

        "reasons": reasons
    }

In [ ]:
def generate_pricing_decision(
    product_data,
    model,
    features
):

    best_result, results_df = optimize_price(
        product_data,
        model,
        features
    )

    explanation = explain_pricing_decision(
        current_price=product_data["current_price"],
        recommended_price=best_result["recommended_price"],
        competitor_price=product_data["competitor_price"],
        inventory_level=product_data["inventory_level"],
        predicted_demand=best_result["predicted_demand"]
    )

    decision = {

        "product_id": product_data["product_id"],

        "current_price":
        product_data["current_price"],

        "recommended_price":
        best_result["recommended_price"],

        "predicted_demand":
        best_result["predicted_demand"],

        "expected_revenue":
        best_result["expected_revenue"],

        "expected_profit":
        best_result["expected_profit"],

        "action":
        explanation["action"],

        "price_change_percentage":
        explanation["price_change_percentage"],

        "reasons":
        explanation["reasons"]
    }

    return decision, results_df

In [ ]:
for i in range(20):

    event = generate_event(
        product_state["product_id"]
    )

    product_state = update_product_state(
        product_state,
        event
    )

    decision, _ = generate_pricing_decision(
        product_state,
        model,
        features
    )

    print("=" * 50)

    print("EVENT:", event["event_type"])

    print(
        "CURRENT PRICE:",
        decision["current_price"]
    )

    print(
        "RECOMMENDED PRICE:",
        decision["recommended_price"]
    )

    print(
        "ACTION:",
        decision["action"]
    )

    print(
        "REASONS:",
        decision["reasons"]
    )